#### Proper Orthogonal Decompostion INTERACTIVE EXPLORER

Explore the POD on parametrised PDE solutions.

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.stats import qmc
import sys
from pathlib import Path

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "boed" / "__init__.py").is_file():
            return p
        if (p / "pyBOED" / "boed" / "__init__.py").is_file():
            return p / "pyBOED"
    return cwd

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for name in list(sys.modules):
    if name == "boed" or name.startswith("boed."):
        del sys.modules[name]

repo_root = PROJECT_ROOT
print(f"Using PROJECT_ROOT={PROJECT_ROOT}")

from boed.reduction.methods import POD
from boed.reduction.methods import DiffusionProblem1D, kappa_smooth, u0_gaussian

In [2]:
# ============================================================================
# WIDGETS
# ============================================================================

# Paramètres de discrétisation
n_spatial_slider = widgets.IntSlider(
    value=100,
    min=50,
    max=500,
    step=50,
    description='# Spatial pts:',
    continuous_update=False
)

n_timesteps_slider = widgets.IntSlider(
    value=50,
    min=20,
    max=200,
    step=10,
    description='# Time steps:',
    continuous_update=False
)

T_final_slider = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=2.0,
    step=0.1,
    description='Final time:',
    continuous_update=False
)

# Paramètres de l'EDP
kappa_mean_slider = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=2.0,
    step=0.1,
    description='κ mean:',
    continuous_update=False
)

kappa_amplitude_slider = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=0.8,
    step=0.05,
    description='κ amplitude:',
    continuous_update=False
)

kappa_frequency_slider = widgets.FloatSlider(
    value=2.0,
    min=0.5,
    max=5.0,
    step=0.5,
    description='κ frequency:',
    continuous_update=False
)

# Condition initiale
z_center_slider = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=0.9,
    step=0.1,
    description='IC center:',
    continuous_update=False
)

width_slider = widgets.FloatSlider(
    value=0.1,
    min=0.02,
    max=0.3,
    step=0.02,
    description='IC width:',
    continuous_update=False
)

amplitude_ic_slider = widgets.FloatSlider(
    value=1.0,
    min=0.5,
    max=2.0,
    step=0.1,
    description='IC amplitude:',
    continuous_update=False
)

# Paramètres POD
n_snapshots_slider = widgets.IntSlider(
    value=50,
    min=10,
    max=200,
    step=10,
    description='# Snapshots:',
    continuous_update=False
)

energy_threshold_slider = widgets.FloatSlider(
    value=0.9999,
    min=0.90,
    max=0.99999,
    step=0.001,
    description='Energy %:',
    continuous_update=False,
    readout_format='.4f'
)

# Paramètres de sampling
sampling_method_dropdown = widgets.Dropdown(
    options=[
        ('Latin Hypercube', 'lhs'),
        ('Random', 'random'),
        ('Sobol', 'sobol')
    ],
    value='lhs',
    description='Sampling:'
)

compute_button_pod = widgets.Button(
    description='Compute POD',
    button_style='success',
    icon='play'
)

# Bouton pour afficher une solution spécifique
show_solution_button = widgets.Button(
    description='Show Single Solution',
    button_style='info',
    icon='eye'
)

output_pod = widgets.Output()

In [3]:
# ============================================================================
# FONCTIONS
# ============================================================================

def generate_parameter_samples(n_samples, method='lhs'):
    """Génère des échantillons de paramètres."""
    
    # Bornes des paramètres
    # [κ_mean, amplitude, frequency, z_center, width, amplitude_IC]
    bounds = np.array([
        [0.1, 2.0],    # κ_mean
        [0.0, 0.8],    # amplitude
        [0.5, 5.0],    # frequency
        [0.2, 0.8],    # z_center
        [0.02, 0.3],   # width
        [0.5, 2.0]     # amplitude_IC
    ])
    
    if method == 'lhs':
        sampler = qmc.LatinHypercube(d=6, seed=42)
        samples_unit = sampler.random(n=n_samples)
        samples = qmc.scale(samples_unit, bounds[:, 0], bounds[:, 1])
    
    elif method == 'random':
        samples = np.random.uniform(bounds[:, 0], bounds[:, 1], (n_samples, 6))
    
    elif method == 'sobol':
        sampler = qmc.Sobol(d=6, seed=42)
        samples_unit = sampler.random(n=n_samples)
        samples = qmc.scale(samples_unit, bounds[:, 0], bounds[:, 1])
    
    return samples


In [4]:
def show_single_solution(button=None):
    """Affiche une solution unique avec les paramètres actuels."""
    
    with output_pod:
        clear_output(wait=True)
        
        print("⏳ Résolution de l'EDP...")
        
        # Créer le problème
        problem = DiffusionProblem1D(
            n_spatial=n_spatial_slider.value,
            T_final=T_final_slider.value,
            n_timesteps=n_timesteps_slider.value
        )
        
        # Paramètres actuels
        params = np.array([
            kappa_mean_slider.value,
            kappa_amplitude_slider.value,
            kappa_frequency_slider.value,
            z_center_slider.value,
            width_slider.value,
            amplitude_ic_slider.value
        ])
        
        # Résoudre
        U = problem.solve(kappa_smooth, u0_gaussian, params)
        
        print("✓ Solution calculée")
        
        # ====================================================================
        # VISUALISATION
        # ====================================================================
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Diffusivity κ(z)',
                'Initial Condition u₀(z)',
                'Solution u(z,t) - Spacetime',
                'Time Evolution at z=0.5'
            ),
            specs=[
                [{'type': 'scatter'}, {'type': 'scatter'}],
                [{'type': 'heatmap'}, {'type': 'scatter'}]
            ]
        )
        
        # SUBPLOT 1 : Diffusivité
        kappa = kappa_smooth(problem.z, params)
        
        fig.add_trace(
            go.Scatter(
                x=problem.z,
                y=kappa,
                mode='lines',
                line=dict(color='blue', width=3),
                name='κ(z)'
            ),
            row=1, col=1
        )
        
        fig.update_xaxes(title_text='z', row=1, col=1)
        fig.update_yaxes(title_text='κ(z)', row=1, col=1)
        
        # SUBPLOT 2 : Condition initiale
        u0 = u0_gaussian(problem.z, params)
        
        fig.add_trace(
            go.Scatter(
                x=problem.z,
                y=u0,
                mode='lines',
                fill='tozeroy',
                line=dict(color='red', width=3),
                name='u₀(z)'
            ),
            row=1, col=2
        )
        
        fig.update_xaxes(title_text='z', row=1, col=2)
        fig.update_yaxes(title_text='u₀(z)', row=1, col=2)
        
        # SUBPLOT 3 : Solution complète (heatmap)
        fig.add_trace(
            go.Heatmap(
                z=U.T,
                x=problem.z,
                y=problem.t,
                colorscale='Jet',
                colorbar=dict(title='u(z,t)', x=1.0)
            ),
            row=2, col=1
        )
        
        fig.update_xaxes(title_text='z (space)', row=2, col=1)
        fig.update_yaxes(title_text='t (time)', row=2, col=1)
        
        # SUBPLOT 4 : Évolution temporelle au centre
        idx_center = n_spatial_slider.value // 2
        
        fig.add_trace(
            go.Scatter(
                x=problem.t,
                y=U[idx_center, :],
                mode='lines+markers',
                line=dict(color='green', width=2),
                marker=dict(size=6),
                name=f'u(z={problem.z[idx_center]:.2f}, t)'
            ),
            row=2, col=2
        )
        
        fig.update_xaxes(title_text='t (time)', row=2, col=2)
        fig.update_yaxes(title_text='u', row=2, col=2)
        
        # Layout
        fig.update_layout(
            height=800,
            title_text='<b>Single PDE Solution</b>',
            showlegend=True
        )
        
        fig.show()
        
        print(f"\nParamètres utilisés :")
        print(f"  κ_mean      = {params[0]:.3f}")
        print(f"  amplitude   = {params[1]:.3f}")
        print(f"  frequency   = {params[2]:.3f}")
        print(f"  z_center    = {params[3]:.3f}")
        print(f"  width       = {params[4]:.3f}")
        print(f"  amplitude_IC = {params[5]:.3f}")

In [5]:
def compute_pod_interactive(button=None):
    """Calcule POD et visualise."""
    
    with output_pod:
        clear_output(wait=True)
        
        print("⏳ Génération des snapshots...")
        
        # Créer le problème
        problem = DiffusionProblem1D(
            n_spatial=n_spatial_slider.value,
            T_final=T_final_slider.value,
            n_timesteps=n_timesteps_slider.value
        )
        
        print(f"  Dimension état : {problem.n_spatial} × {problem.n_timesteps} = {problem.n_spatial * problem.n_timesteps}")
        
        # Générer paramètres
        parameter_samples = generate_parameter_samples(
            n_snapshots_slider.value,
            method=sampling_method_dropdown.value
        )
        
        print(f"  {n_snapshots_slider.value} paramètres générés ({sampling_method_dropdown.label})")
        
        # Générer snapshots
        snapshots = problem.solve_multiple(
            kappa_smooth,
            u0_gaussian,
            parameter_samples
        )
        
        print(f"✓ Snapshots générés : {snapshots.shape}")
        
        print("\n⏳ Construction de la base POD...")
        
        # POD
        pod = POD(energy_threshold=energy_threshold_slider.value)
        pod.fit(snapshots, use_snapshot_method=True)
        
        print(f"✓ POD calculé : {pod.n_components} modes")
        
        # ====================================================================
        # VISUALISATION
        # ====================================================================
        
        fig = make_subplots(
            rows=3, cols=2,
            subplot_titles=(
                'POD Eigenvalues (Energy)',
                'Cumulative Energy',
                'First 4 POD Modes',
                'Reconstruction Error vs Rank',
                'Mode Amplitudes Distribution',
                'Snapshot Projection (PC1 vs PC2)'
            ),
            specs=[
                [{'type': 'scatter'}, {'type': 'scatter'}],
                [{'type': 'scatter'}, {'type': 'scatter'}],
                [{'type': 'box'}, {'type': 'scatter'}]
            ],
            vertical_spacing=0.12,
            horizontal_spacing=0.15
        )
        
        # SUBPLOT 1 : Eigenvalues
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(pod.eigenvalues_) + 1)),
                y=pod.eigenvalues_,
                mode='markers+lines',
                marker=dict(size=8, color='blue'),
                name='Energy λᵢ'
            ),
            row=1, col=1
        )
        
        fig.add_vline(x=pod.n_components, line_dash='dash', line_color='red',
                      annotation_text=f'r={pod.n_components}', row=1, col=1)
        
        fig.update_yaxes(type='log', title_text='Eigenvalue', row=1, col=1)
        fig.update_xaxes(title_text='Mode', row=1, col=1)
        
        # SUBPLOT 2 : Cumulative energy
        cumsum = np.cumsum(pod.energy_ratio_) * 100
        
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(cumsum) + 1)),
                y=cumsum,
                mode='lines',
                fill='tozeroy',
                line=dict(color='green', width=3),
                name='Cumulative'
            ),
            row=1, col=2
        )
        
        fig.add_hline(y=energy_threshold_slider.value * 100,
                      line_dash='dash', line_color='red', row=1, col=2)
        
        fig.update_yaxes(title_text='Energy (%)', row=1, col=2)
        fig.update_xaxes(title_text='Modes', row=1, col=2)
        
        # SUBPLOT 3 : Premiers modes POD
        # Reshape modes pour visualisation spacetime
        n_modes_show = min(4, pod.n_components)
        
        import plotly.express as px
        colors = px.colors.qualitative.Set2
        
        for i in range(n_modes_show):
            mode = pod.basis_[:, i].reshape(problem.n_spatial, problem.n_timesteps, order='F')
            # Afficher mode au temps final
            fig.add_trace(
                go.Scatter(
                    x=problem.z,
                    y=mode[:, -1],
                    mode='lines',
                    name=f'Mode {i+1}',
                    line=dict(color=colors[i % len(colors)], width=2)
                ),
                row=2, col=1
            )
        
        fig.update_xaxes(title_text='z (space)', row=2, col=1)
        fig.update_yaxes(title_text='φᵢ(z, T_final)', row=2, col=1)
        
        # SUBPLOT 4 : Erreur de reconstruction
        errors = []
        ranks = range(1, min(30, len(pod.eigenvalues_)))
        
        for r in ranks:
            V_r = pod.basis_[:, :r]
            snapshots_r = V_r @ (V_r.T @ snapshots)
            error = np.linalg.norm(snapshots - snapshots_r, 'fro') / np.linalg.norm(snapshots, 'fro')
            errors.append(error)
        
        # Theoretical error
        theoretical_errors = []
        for r in ranks:
            residual_energy = pod.eigenvalues_[r:].sum()
            total_energy = pod.eigenvalues_.sum()
            theoretical_errors.append(np.sqrt(residual_energy / total_energy))
        
        fig.add_trace(
            go.Scatter(
                x=list(ranks),
                y=errors,
                mode='markers+lines',
                marker=dict(size=6, color='blue'),
                name='Empirical'
            ),
            row=2, col=2
        )
        
        fig.add_trace(
            go.Scatter(
                x=list(ranks),
                y=theoretical_errors,
                mode='lines',
                line=dict(dash='dash', color='red', width=2),
                name='Theoretical'
            ),
            row=2, col=2
        )
        
        fig.update_yaxes(type='log', title_text='Relative Error', row=2, col=2)
        fig.update_xaxes(title_text='Rank', row=2, col=2)
        
        # SUBPLOT 5 : Distribution des amplitudes modales
        coords = pod.transform(snapshots)
        
        for i in range(min(5, pod.n_components)):
            fig.add_trace(
                go.Box(
                    y=coords[i, :],
                    name=f'Mode {i+1}',
                    boxmean='sd'
                ),
                row=3, col=1
            )
        
        fig.update_yaxes(title_text='Amplitude', row=3, col=1)
        
        # SUBPLOT 6 : Projection 2D
        if pod.n_components >= 2:
            fig.add_trace(
                go.Scatter(
                    x=coords[0, :],
                    y=coords[1, :],
                    mode='markers',
                    marker=dict(
                        size=6,
                        color=np.arange(coords.shape[1]),
                        colorscale='Viridis',
                        showscale=True,
                        colorbar=dict(title='Snapshot #')
                    ),
                    name='Snapshots'
                ),
                row=3, col=2
            )
            
            fig.update_xaxes(title_text='Mode 1 amplitude', row=3, col=2)
            fig.update_yaxes(title_text='Mode 2 amplitude', row=3, col=2)
        
        # Layout
        fig.update_layout(
            height=1200,
            showlegend=True,
            title_text=f'<b>POD Analysis</b> ({n_snapshots_slider.value} snapshots, {sampling_method_dropdown.label})'
        )
        
        fig.show()
        
        # ====================================================================
        # STATISTIQUES
        # ====================================================================
        
        print("\n" + "="*70)
        print("STATISTIQUES")
        print("="*70)
        print(f"Dimension état complète : {snapshots.shape[0]}")
        print(f"Nombre de snapshots     : {snapshots.shape[1]}")
        print(f"Dimension réduite       : {pod.n_components}")
        print(f"Compression             : {snapshots.shape[0] / pod.n_components:.1f}×")
        print(f"Énergie capturée        : {cumsum[pod.n_components-1]:.4f}%")
        
        print(f"\nPremiers eigenvalues :")
        for i in range(min(10, len(pod.eigenvalues_))):
            print(f"  λ_{i+1} = {pod.eigenvalues_[i]:.6e} ({pod.energy_ratio_[i]*100:.4f}%)")
        
        # Test reconstruction
        test_snapshot = snapshots[:, 0]
        test_coords = pod.transform(test_snapshot)
        test_recon = pod.inverse_transform(test_coords)
        recon_error = np.linalg.norm(test_snapshot - test_recon) / np.linalg.norm(test_snapshot)
        
        print(f"\nTest reconstruction (snapshot 0) :")
        print(f"  Erreur relative : {recon_error:.4e}")


compute_button_pod.on_click(compute_pod_interactive)
show_solution_button.on_click(show_single_solution)



In [6]:
# ============================================================================
# INTERFACE
# ============================================================================

discretization_box = widgets.VBox([
    widgets.HTML("<h3>🔢 Discretization</h3>"),
    n_spatial_slider,
    n_timesteps_slider,
    T_final_slider
])

pde_params_box = widgets.VBox([
    widgets.HTML("<h3>🌡️ PDE Parameters</h3>"),
    widgets.HTML("<b>Diffusivity κ(z):</b>"),
    kappa_mean_slider,
    kappa_amplitude_slider,
    kappa_frequency_slider,
    widgets.HTML("<b>Initial condition u₀(z):</b>"),
    z_center_slider,
    width_slider,
    amplitude_ic_slider,
    show_solution_button
])

pod_params_box = widgets.VBox([
    widgets.HTML("<h3>🎯 POD Parameters</h3>"),
    n_snapshots_slider,
    sampling_method_dropdown,
    energy_threshold_slider,
    compute_button_pod
])

controls_pod = widgets.HBox([discretization_box, pde_params_box, pod_params_box])

display(widgets.VBox([
    widgets.HTML("<h1>🔥 POD Interactive Explorer</h1>"),
    widgets.HTML("<p>Explore Proper Orthogonal Decomposition on parametric diffusion PDEs</p>"),
    controls_pod,
    output_pod
]))